In [1]:
import pandas as pd
import openpyxl

In [2]:
path = r"H:\python\3atapa\CD2022_Populacao_2010_Compatibilizada_20231222 (1).xlsx"

In [3]:
todas = pd.read_excel(path, sheet_name=None, header=None)
print(todas.keys())

dict_keys(['Municípios'])


In [7]:
bruto = pd.read_excel(path, sheet_name="Municípios", header=None)
bruto

,0,1,2,3,4,5,6,7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Censo Demográfico 2022: População e Domicílios...,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,População Município 2010\n(Sinopse),População 2010 (Alterações de Limites até 2022)1,População Censo 2022
3,NaN,RO,11,00015,Alta Floresta D'Oeste,24392,24392,21494
4,NaN,RO,11,00023,Ariquemes,90353,90353,96833
...,...,...,...,...,...,...,...,...
5572,NaN,DF,53,00108,Brasília,2570160,2572159,2817381
5573,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5574,NaN,Nota: Para o cálculo das taxas de crescimento ...,NaN,NaN,NaN,NaN,NaN,NaN
5575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
populacao = bruto.iloc[3:5573].copy()
populacao

,0,1,2,3,4,5,6,7
3,NaN,RO,11,00015,Alta Floresta D'Oeste,24392,24392,21494
4,NaN,RO,11,00023,Ariquemes,90353,90353,96833
5,NaN,RO,11,00031,Cabixi,6313,6313,5351
6,NaN,RO,11,00049,Cacoal,78574,78574,86887
7,NaN,RO,11,00056,Cerejeiras,17029,17029,15890
...,...,...,...,...,...,...,...,...
5568,NaN,GO,52,22005,Vianópolis,12548,12548,14956
5569,NaN,GO,52,22054,Vicentinópolis,7371,7373,8768
5570,NaN,GO,52,22203,Vila Boa,4735,4735,4215
5571,NaN,GO,52,22302,Vila Propício,5145,5145,5815


In [ ]:
populacao.columns = [
    "col_vazia",
    "uf",
    "cod_uf",
    "cod_munic",
    "municipio",
    "pop_2010_sinopse",
    "pop_2010_compat",
    "pop_2022",
]

populacao = populacao.drop(columns=["col_vazia"])
populacao = populacao.reset_index(drop=True)
populacao

,uf,cod_uf,cod_munic,municipio,pop_2010_sinopse,pop_2010_compat,pop_2022
0,RO,11,00015,Alta Floresta D'Oeste,24392,24392,21494
1,RO,11,00023,Ariquemes,90353,90353,96833
2,RO,11,00031,Cabixi,6313,6313,5351
3,RO,11,00049,Cacoal,78574,78574,86887
4,RO,11,00056,Cerejeiras,17029,17029,15890
...,...,...,...,...,...,...,...
5565,GO,52,22005,Vianópolis,12548,12548,14956
5566,GO,52,22054,Vicentinópolis,7371,7373,8768
5567,GO,52,22203,Vila Boa,4735,4735,4215
5568,GO,52,22302,Vila Propício,5145,5145,5815


In [11]:
for col in ["pop_2010_sinopse", "pop_2010_compat", "pop_2022"]:
    populacao[col] = pd.to_numeric(pop[col], errors="coerce")

populacao.describe().round(2)

,pop_2010_sinopse,pop_2010_compat,pop_2022
count,5570.00,5570.00,5570.00
mean,34247.00,34247.00,36459.74
std,203024.02,203037.82,206518.73
min,0.00,805.00,833.00
25%,5224.00,5236.50,5228.00
50%,10928.50,10896.50,11065.00
75%,23409.00,23413.00,24427.25
max,11253503.00,11253503.00,11451999.00


In [12]:
populacao[populacao["uf"].isna() | pop["pop_2022"].isna()]

,uf,cod_uf,cod_munic,municipio,pop_2010_sinopse,pop_2010_compat,pop_2022


In [13]:
populacao_estado = populacao.groupby("uf", as_index=False)[["pop_2010_compat", "pop_2022"]].sum()
populacao_estado

,uf,pop_2010_compat,pop_2022
0,AC,733559,830018
1,AL,3120887,3127683
2,AM,3483985,3941613
3,AP,669526,733759
4,BA,14017071,14141626
5,CE,8451644,8794957
6,DF,2572159,2817381
7,ES,3514952,3833712
8,GO,6001789,7056495
9,MA,6574789,6776699


In [14]:
populacao_estado["diferenca"] = populacao_estado["pop_2022"] - populacao_estado["pop_2010_compat"]
populacao_estado["cresc_pct"] = (populacao_estado["diferenca"] / populacao_estado["pop_2010_compat"] * 100).round(2)

populacao_estado

,uf,pop_2010_compat,pop_2022,diferenca,cresc_pct
0,AC,733559,830018,96459,13.15
1,AL,3120887,3127683,6796,0.22
2,AM,3483985,3941613,457628,13.14
3,AP,669526,733759,64233,9.59
4,BA,14017071,14141626,124555,0.89
5,CE,8451644,8794957,343313,4.06
6,DF,2572159,2817381,245222,9.53
7,ES,3514952,3833712,318760,9.07
8,GO,6001789,7056495,1054706,17.57
9,MA,6574789,6776699,201910,3.07


In [15]:
populacao_estado = populacao_estado.sort_values("diferenca", ascending=False).reset_index(drop=True)
populacao_estado

,uf,pop_2010_compat,pop_2022,diferenca,cresc_pct
0,SP,41262199,44411238,3149039,7.63
1,SC,6248436,7610361,1361925,21.80
2,GO,6001789,7056495,1054706,17.57
3,PR,10444526,11444380,999854,9.57
4,MG,19597330,20539989,942659,4.81
5,MT,3035122,3658649,623527,20.54
6,PA,7581051,8120131,539080,7.11
7,AM,3483985,3941613,457628,13.14
8,CE,8451644,8794957,343313,4.06
9,ES,3514952,3833712,318760,9.07


In [16]:
populacao_estado.to_csv("populacao_por_estado_2010_2022.csv", index=False, encoding="utf-8-sig")
print("salvo: populacao_por_estado_2010_2022.csv")


salvo: populacao_por_estado_2010_2022.csv


In [17]:
populacao_estado = populacao[["uf", "cod_uf", "cod_munic", "municipio", "pop_2010_compat", "pop_2022"]].copy()
populacao_estado


,uf,cod_uf,cod_munic,municipio,pop_2010_compat,pop_2022
0,RO,11,00015,Alta Floresta D'Oeste,24392,21494
1,RO,11,00023,Ariquemes,90353,96833
2,RO,11,00031,Cabixi,6313,5351
3,RO,11,00049,Cacoal,78574,86887
4,RO,11,00056,Cerejeiras,17029,15890
...,...,...,...,...,...,...
5565,GO,52,22005,Vianópolis,12548,14956
5566,GO,52,22054,Vicentinópolis,7373,8768
5567,GO,52,22203,Vila Boa,4735,4215
5568,GO,52,22302,Vila Propício,5145,5815


In [18]:
populacao_estado["diferenca"] = populacao_estado["pop_2022"] - populacao_estado["pop_2010_compat"]
populacao_estado["cresc_pct"] = (populacao_estado["diferenca"] / populacao_estado["pop_2010_compat"] * 100).round(2)

populacao_estado

,uf,cod_uf,cod_munic,municipio,pop_2010_compat,pop_2022,diferenca,cresc_pct
0,RO,11,00015,Alta Floresta D'Oeste,24392,21494,-2898,-11.88
1,RO,11,00023,Ariquemes,90353,96833,6480,7.17
2,RO,11,00031,Cabixi,6313,5351,-962,-15.24
3,RO,11,00049,Cacoal,78574,86887,8313,10.58
4,RO,11,00056,Cerejeiras,17029,15890,-1139,-6.69
...,...,...,...,...,...,...,...,...
5565,GO,52,22005,Vianópolis,12548,14956,2408,19.19
5566,GO,52,22054,Vicentinópolis,7373,8768,1395,18.92
5567,GO,52,22203,Vila Boa,4735,4215,-520,-10.98
5568,GO,52,22302,Vila Propício,5145,5815,670,13.02


In [19]:
populacao_estado = populacao_estado.sort_values("diferenca", ascending=False).reset_index(drop=True)
populacao_estado.head(20)

,uf,cod_uf,cod_munic,municipio,pop_2010_compat,pop_2022,diferenca,cresc_pct
0,AM,13,02603,Manaus,1802014,2063689,261675,14.52
1,DF,53,00108,Brasília,2572159,2817381,245222,9.53
2,SP,35,50308,São Paulo,11253503,11451999,198496,1.76
3,SP,35,52205,Sorocaba,586816,723682,136866,23.32
4,GO,52,08707,Goiânia,1301912,1437366,135454,10.40
5,RR,14,00100,Boa Vista,284313,413486,129173,45.43
6,SC,42,05407,Florianópolis,421240,537211,115971,27.53
7,PA,15,05536,Parauapebas,153908,267836,113928,74.02
8,MS,50,02704,Campo Grande,786774,898100,111326,14.15
9,PB,25,07507,João Pessoa,723515,833932,110417,15.26


In [20]:
populacao_estado.sort_values("cresc_pct", ascending=False).head(10)

,uf,cod_uf,cod_munic,municipio,pop_2010_compat,pop_2022,diferenca,cresc_pct
56,PA,15,02152,Canaã dos Carajás,26716,77079,50363,188.51
286,GO,52,00050,Abadia de Goiás,6891,19128,12237,177.58
85,RN,24,03608,Extremoz,24569,61635,37066,150.86
84,GO,52,08806,Goianira,34058,71916,37858,111.16
220,SC,42,08450,Itapoá,14763,30750,15987,108.29
251,MT,51,07065,Querência,13033,26769,13736,105.39
151,SC,42,02107,Barra Velha,22386,45369,22983,102.67
294,AL,27,08907,Satuba,12443,24278,11835,95.11
505,SC,42,12254,Passo de Torres,6627,12897,6270,94.61
443,SC,42,02073,Balneário Gaivota,8234,15669,7435,90.30


In [21]:
populacao_estado.to_csv("populacao_por_municipio_2010_2022.csv", index=False, encoding="utf-8-sig")
print("salvo: populacao_por_municipio_2010_2022.csv")

salvo: populacao_por_municipio_2010_2022.csv
